In [105]:
import numpy as np
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.stem.porter import PorterStemmer

In [107]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')

In [108]:
movies=movies.merge(credits,on='title')

In [109]:
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [110]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [111]:
movies.dropna(inplace=True)

In [112]:
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [113]:
movies['genres']=movies['genres'].apply(convert)

In [114]:
movies['keywords']=movies['keywords'].apply(convert)

In [115]:
def convert3(obj):
    L=[]
    cnt=0
    for i in ast.literal_eval(obj):
        if cnt!=3:
            L.append(i['name'])
            cnt+=1
        else:
            break;
    return L

In [116]:
movies['cast']=movies['cast'].apply(convert3)

In [117]:
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

In [118]:
movies['crew']=movies['crew'].apply(fetch_director)

In [119]:
movies['overview']=movies['overview'].apply(lambda a:a.split())

In [120]:
movies['genres']=movies['genres'].apply(lambda a:[i.replace(" ","")for i in a])
movies['keywords']=movies['keywords'].apply(lambda a:[i.replace(" ","")for i in a])
movies['cast']=movies['cast'].apply(lambda a:[i.replace(" ","")for i in a])
movies['crew']=movies['crew'].apply(lambda a:[i.replace(" ","")for i in a])

In [121]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [122]:
new_df=movies[['movie_id','title','tags']]

In [123]:
new_df['tags']=new_df['tags'].apply(lambda a:" ".join(a))

C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_7268\482793026.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda a:" ".join(a))


In [124]:
new_df['tags']=new_df['tags'].apply(lambda a:a.lower())

C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_7268\182077803.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda a:a.lower())


In [125]:
ps=PorterStemmer()
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)



new_df['tags']=new_df['tags'].apply(stem)

C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_7268\1413720578.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(stem)


In [129]:
cv=CountVectorizer(max_features=5000,stop_words='english')

In [133]:
vectors=cv.fit_transform(new_df['tags']).toarray()

In [153]:
from sklearn.metrics.pairwise import cosine_similarity

In [173]:
similarity=cosine_similarity(vectors)

In [185]:
def recommend(movie):
    movie_ind=new_df[new_df['title']==movie].index[0]
    distances=similarity[movie_ind]
    movies_list=sorted(list(enumerate(distances)),reverse=True,key=lambda a:a[1])[1:6]
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [193]:
recommend('Titanic')

The Notebook
Under the Same Moon
Ghost Ship
The Bounty
Pirates of the Caribbean: On Stranger Tides


In [195]:
import pickle

In [199]:
pickle.dump(new_df.to_dict(),open('movie_dict.pkl','wb'))

In [201]:
pickle.dump(similarity,open('similarity.pkl','wb'))